# 03 — ZOO Attack (Zeroth-Order Optimisation)

**Goal:** Suppress YOLO11 confidence scores via SPSA (random-direction finite-difference) gradient estimation.
**Access:** Greybox — confidence scores only. No model weights, no backprop.
**Target:** Reduce detection confidence below YOLO's threshold (0.25) → object disappears.
**Monitor signature:** High-volume burst of near-identical images (pHash Hamming 0–4).

### How SPSA estimates gradients without model access
```
Each iteration:
    sample random direction d (±1 per pixel/channel in the patch)
    query(image + δ·d)  → conf_plus
    query(image - δ·d)  → conf_minus
    gradient ≈ (conf_plus - conf_minus) / (2δ) · d
    patch update: patch -= lr × gradient

Two API queries per iteration, regardless of patch size.
Attack a small patch (e.g. 32×32) not the full image — keeps the
perturbation localised and the monitor signature (near-identical
repeated queries) realistic.
```

### Run 001 retrospective (superseded)
First run used per-pixel finite differences (DELTA=1.0, LR=2.0):
2×PATCH² = 6,144 queries/iteration. After 50 iterations (307,200
queries) confidence moved 0.9179 → 0.9171 — effectively no change.
±1/255 pixel perturbations are below YOLO's confidence sensitivity
floor, so the gradient estimate was pure noise. Switched to SPSA with
a much larger step size (DELTA=16) and higher LR — 2 queries/iteration
instead of thousands.

### Run 002 retrospective (superseded)
Switched to SPSA (random-direction finite differences), `DELTA=16`,
`LR=16`, 300 iterations (600 queries). Confidence moved 0.9179 → 0.9177
(Δ=0.00023); patch pixels changed by mean 0.6, max 5/255 — sub-pixel
updates per iteration. The gradient-magnitude-scaled update rule
(`update = LR/(2·DELTA) · conf_diff · direction`) is mis-scaled: with
`conf_diff ~ 0.001`, each step is ~1/2000th of a pixel level. Direction was
roughly correct but step size never accumulated to anything within budget.

Switched to a greedy fixed-step search (SimBA-style): each iteration try
`±DELTA·d` for a random direction `d`, commit the *full* `DELTA` step
whichever direction reduces confidence (or skip if neither helps). Step
size is now decoupled from gradient magnitude — every successful iteration
makes real pixel-level progress.

### Run 004 plan (current)
New target image `data/one car traffic light.png` (1280x653, 1 car @ 0.771,
traffic lights up to 0.611) — simpler scene than gettyimage, and the
traffic light (class 9, conf=0.611) is much closer to the 0.25 detection
threshold than any car was.

Per the priority realignment: the goal is a bounded, representative A2
(ZOO) signature log — true per-pixel coordinate probing (2 queries per
pixel-channel, near-identical images, pHash≈0 burst), not a 307K-query
run. Combines:
- **ZOO's query pattern**: sequential per-pixel ±DELTA coordinate probes
  (the actual A2 signature).
- **Greedy commit** (learned from Run 003): each coordinate's update is
  committed at the *full* DELTA step if it improves confidence, not
  scaled by gradient magnitude — avoids the sub-pixel-drift failure of
  Runs 001/002.

`PATCH_SIZE=8` → 192 pixel-channels/pass, 384 queries/pass, `MAX_ITER=8`
passes → up to ~3,072 queries. Target = traffic light (conf=0.611,
class 9) first; the car (conf=0.771, class 2) is a second target to try
afterward by changing `TARGET_CLASS`.


## 0. Config

In [ ]:
from pathlib import Path

# ── Edit these ───────────────────────────────────────────────
MODEL_PATH     = Path("../models/yolo11n.pt")
LOG_FILE       = Path("../logs/query_log_v2.jsonl")   # baseline log (read-only: query_id continuity + normal-traffic stats)
ATTACK_LOG_DIR = Path("../logs/attacks")              # gitignored — attack queries go here, never the baseline log
BASE_IMAGE     = Path("../data/one car traffic light.png")  # 1 car + traffic lights, less crowded
VIDEO_PATH     = "../data/dash-cam-video.mp4"
OUTPUT_DIR     = Path("../data/zoo_outputs")

SESSION_ID      = "attack_zoo_002"   # unique per attack run
ATTACK_LOG_FILE = ATTACK_LOG_DIR / f"{SESSION_ID}.jsonl"
TARGET_CLASS    = 9                  # COCO class 9 = traffic light (conf=0.611, closer to 0.25 threshold than any car)

# ZOO hyperparameters (per-pixel coordinate probing, greedy fixed-step commit)
PATCH_SIZE   = 8      # pixels — attack region side length (8x8 = 192 pixel-channels)
DELTA        = 8.0    # fixed step size per successful coordinate (pixel units, 0-255 scale)
MAX_ITER     = 8      # passes over all pixel-channels (each pass = 2*PATCH_SIZE^2*3 queries)
CONF_TARGET  = 0.20   # stop if max conf for target class drops below this
# ─────────────────────────────────────────────────────────────

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ATTACK_LOG_DIR.mkdir(parents=True, exist_ok=True)
print(f"Session    : {SESSION_ID}")
print(f"Attack log : {ATTACK_LOG_FILE}")
print(f"Target     : COCO class {TARGET_CLASS}")
print(f"Patch      : {PATCH_SIZE}\u00d7{PATCH_SIZE} px ({PATCH_SIZE*PATCH_SIZE*3} pixel-channels)")
print(f"Max passes : {MAX_ITER}")
print(f"Max queries this run : ~{MAX_ITER * 2 * PATCH_SIZE*PATCH_SIZE*3:,}  (per-pixel ZOO, greedy commit)")


## 1. Load Dependencies + Wrapper

In [10]:
import hashlib, json, time
import numpy as np
import cv2
import imagehash as ih
from PIL import Image
from ultralytics import YOLO

model = YOLO(str(MODEL_PATH))

# ── Persistent query_id (continues across baseline + all attack logs) ──
def _last_qid_in(path):
    if not path.exists():
        return 0
    lines = path.read_text().strip().splitlines()
    return json.loads(lines[-1])["query_id"] if lines else 0

def _load_last_qid():
    candidates = [LOG_FILE] + sorted(ATTACK_LOG_DIR.glob("*.jsonl"))
    return max((_last_qid_in(p) for p in candidates), default=0)

_global_query_id = _load_last_qid()
_session_last_ts = {}
print(f"query_id continues from: {_global_query_id + 1}")

def _to_pil(img_array):
    return Image.fromarray(cv2.cvtColor(img_array, cv2.COLOR_BGR2RGB))

def query(image_array, source_label="attack_zoo"):
    """Greybox API — same log format as wrapper v2. Writes to ATTACK_LOG_FILE (gitignored)."""
    global _global_query_id
    _global_query_id += 1
    now = time.time()

    last_ts  = _session_last_ts.get(SESSION_ID)
    delta_ms = round((now - last_ts) * 1000, 1) if last_ts else None
    _session_last_ts[SESSION_ID] = now

    phash = str(ih.phash(_to_pil(image_array)))
    md5   = hashlib.md5(image_array.tobytes()).hexdigest()

    t0 = time.perf_counter()
    results    = model(image_array, verbose=False)
    latency_ms = round((time.perf_counter() - t0) * 1000, 1)

    detections = [{
        "class":      int(box.cls[0]),
        "class_name": model.names[int(box.cls[0])],
        "conf":       round(float(box.conf[0]), 4),
        "bbox":       [round(x, 2) for x in box.xyxy[0].tolist()]
    } for box in results[0].boxes]

    entry = {
        "query_id":     _global_query_id,
        "session_id":   SESSION_ID,
        "timestamp":    now,
        "delta_ms":     delta_ms,
        "source":       source_label,
        "md5":          md5,
        "phash":        phash,
        "n_detections": len(detections),
        "latency_ms":   latency_ms,
        "detections":   detections
    }
    with open(ATTACK_LOG_FILE, "a") as f:
        f.write(json.dumps(entry) + "\n")

    return detections


query_id continues from: 37687


## 2. Select Target — Find Best Detection to Attack

In [11]:
# Load base image
base_img = cv2.imread(str(BASE_IMAGE))
assert base_img is not None, f"Cannot load: {BASE_IMAGE}"
H, W = base_img.shape[:2]
print(f"Image: {W}×{H}")

# Clean baseline query (not logged as attack)
baseline_results = model(base_img, verbose=False)
detections = []
for box in baseline_results[0].boxes:
    cls  = int(box.cls[0])
    conf = float(box.conf[0])
    bbox = [round(x, 1) for x in box.xyxy[0].tolist()]
    detections.append({"class": cls, "name": model.names[cls], "conf": conf, "bbox": bbox})

print(f"\nAll detections:")
for i, d in enumerate(detections):
    marker = " ← TARGET" if d["class"] == TARGET_CLASS else ""
    print(f"  [{i}] {d['name']:12s} conf={d['conf']:.3f}  bbox={d['bbox']}{marker}")

# Pick highest-conf target-class detection
targets = [d for d in detections if d["class"] == TARGET_CLASS]
if not targets:
    raise ValueError(f"No class {TARGET_CLASS} ({model.names[TARGET_CLASS]}) detected. Change TARGET_CLASS.")

target = max(targets, key=lambda d: d["conf"])
print(f"\nAttacking: {target['name']} conf={target['conf']:.3f} bbox={target['bbox']}")

# Place patch at centre of target bbox
x1, y1, x2, y2 = [int(v) for v in target["bbox"]]
cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
px1 = max(0, cx - PATCH_SIZE // 2)
py1 = max(0, cy - PATCH_SIZE // 2)
px2 = min(W, px1 + PATCH_SIZE)
py2 = min(H, py1 + PATCH_SIZE)
print(f"Patch region: ({px1},{py1}) → ({px2},{py2})")

Image: 768×432

All detections:
  [0] car          conf=0.918  bbox=[200.6, 236.4, 418.6, 394.0] ← TARGET
  [1] car          conf=0.881  bbox=[600.7, 222.7, 743.5, 334.9] ← TARGET
  [2] truck        conf=0.798  bbox=[70.7, 147.4, 240.0, 311.4]
  [3] car          conf=0.773  bbox=[0.0, 205.6, 172.4, 348.1] ← TARGET
  [4] car          conf=0.762  bbox=[318.6, 217.3, 446.8, 332.3] ← TARGET
  [5] car          conf=0.739  bbox=[0.1, 278.2, 36.0, 377.7] ← TARGET
  [6] car          conf=0.700  bbox=[487.3, 199.9, 530.0, 243.1] ← TARGET
  [7] car          conf=0.693  bbox=[450.7, 206.0, 502.8, 282.9] ← TARGET
  [8] car          conf=0.657  bbox=[352.2, 196.0, 471.0, 300.7] ← TARGET
  [9] car          conf=0.620  bbox=[571.1, 219.6, 616.7, 269.7] ← TARGET
  [10] car          conf=0.388  bbox=[226.0, 197.4, 301.4, 249.1] ← TARGET
  [11] car          conf=0.305  bbox=[277.1, 191.0, 336.8, 238.0] ← TARGET
  [12] car          conf=0.270  bbox=[575.1, 199.5, 641.8, 229.8] ← TARGET

Attacking: car co

## 3. Helper — Get Target Confidence

In [12]:
def get_target_conf(image_array, log=True):
    """
    Query API, return max confidence for TARGET_CLASS.
    Returns 0.0 if class not detected (attack succeeded).
    """
    dets = query(image_array) if log else [
        {"class": int(b.cls[0]), "conf": float(b.conf[0])}
        for b in model(image_array, verbose=False)[0].boxes
    ]
    confs = [d["conf"] for d in dets if d["class"] == TARGET_CLASS]
    return max(confs) if confs else 0.0

## 4. ZOO Attack Loop

In [ ]:
adv_img   = base_img.astype(np.float32).copy()
conf_log  = []   # track conf after each coordinate decision
query_log = []   # track cumulative queries
total_queries = 0

patch_h = py2 - py1
patch_w = px2 - px1
n_coords = patch_h * patch_w * 3

init_conf = get_target_conf(base_img.astype(np.uint8), log=False)
current_conf = init_conf
print(f"Initial confidence: {init_conf:.4f}")
print(f"Target threshold  : {CONF_TARGET}")
print(f"Patch pixels      : {patch_h * patch_w} ({n_coords} pixel-channels)")
print(f"Queries per pass  : {2 * n_coords:,}")
print(f"Max passes        : {MAX_ITER}")
print()

stop = False
for pass_num in range(MAX_ITER):
    moved_count = 0
    for pi in range(patch_h):
        for pj in range(patch_w):
            for ch in range(3):
                y, x = py1 + pi, px1 + pj
                orig_val = adv_img[y, x, ch]

                # Probe +DELTA
                img_plus = adv_img.copy()
                img_plus[y, x, ch] = np.clip(orig_val + DELTA, 0, 255)
                conf_plus = get_target_conf(img_plus.astype(np.uint8))
                total_queries += 1

                # Probe -DELTA
                img_minus = adv_img.copy()
                img_minus[y, x, ch] = np.clip(orig_val - DELTA, 0, 255)
                conf_minus = get_target_conf(img_minus.astype(np.uint8))
                total_queries += 1

                # Greedy: commit the full step if either probe improves on current
                best_conf, best_val = current_conf, None
                if conf_plus < best_conf:
                    best_conf, best_val = conf_plus, img_plus[y, x, ch]
                if conf_minus < best_conf:
                    best_conf, best_val = conf_minus, img_minus[y, x, ch]

                if best_val is not None:
                    adv_img[y, x, ch] = best_val
                    current_conf = best_conf
                    moved_count += 1

                conf_log.append(current_conf)
                query_log.append(total_queries)

                if current_conf < CONF_TARGET or current_conf <= 0.0:
                    stop = True
                    break
            if stop:
                break
        if stop:
            break

    print(f"Pass {pass_num+1:2d}/{MAX_ITER} | conf={current_conf:.4f} | "
          f"moved {moved_count:3d}/{n_coords} coords | queries={total_queries:,}", end="")

    if stop:
        if current_conf < CONF_TARGET:
            print(f"  \u2705 SUCCESS \u2014 below threshold {CONF_TARGET}")
        else:
            print(f"  \u2705 TARGET CLASS GONE")
        break
    else:
        print()

print(f"\nFinal confidence : {conf_log[-1]:.4f}  (started: {init_conf:.4f})")
print(f"Total queries    : {total_queries:,}")
print(f"Conf drop        : {init_conf - conf_log[-1]:.4f}")


## 5. Save Outputs

In [15]:
adv_uint8 = adv_img.astype(np.uint8)

# Save adversarial image
adv_path = OUTPUT_DIR / f"zoo_adv_{SESSION_ID}.jpg"
cv2.imwrite(str(adv_path), adv_uint8)

# Save side-by-side comparison
compare = np.hstack([base_img, adv_uint8])
# Draw patch box on both
for img_c in [compare[:, :W], compare[:, W:]]:
    cv2.rectangle(img_c, (px1, py1), (px2, py2), (0, 0, 255), 2)
compare_path = OUTPUT_DIR / f"zoo_compare_{SESSION_ID}.jpg"
cv2.imwrite(str(compare_path), compare)

# Save perturbation (amplified ×10 for visibility)
diff = np.abs(adv_img - base_img.astype(np.float32))
diff_vis = np.clip(diff * 10, 0, 255).astype(np.uint8)
diff_path = OUTPUT_DIR / f"zoo_diff_{SESSION_ID}.jpg"
cv2.imwrite(str(diff_path), diff_vis)

# Save attack metadata
meta = {
    "session_id":    SESSION_ID,
    "target_class":  TARGET_CLASS,
    "target_name":   model.names[TARGET_CLASS],
    "init_conf":     init_conf,
    "final_conf":    float(conf_log[-1]),
    "conf_drop":     float(init_conf - conf_log[-1]),
    "total_queries": total_queries,
    "iterations":    len(conf_log),
    "patch_region":  [px1, py1, px2, py2],
    "conf_log":      [float(c) for c in conf_log],
    "query_log":     query_log,
    "success":       conf_log[-1] < CONF_TARGET
}
meta_path = OUTPUT_DIR / f"zoo_meta_{SESSION_ID}.json"
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print(f"Adversarial image : {adv_path}")
print(f"Comparison        : {compare_path}")
print(f"Perturbation      : {diff_path}")
print(f"Metadata          : {meta_path}")

Adversarial image : ../data/zoo_outputs/zoo_adv_attack_zoo_001.jpg
Comparison        : ../data/zoo_outputs/zoo_compare_attack_zoo_001.jpg
Perturbation      : ../data/zoo_outputs/zoo_diff_attack_zoo_001.jpg
Metadata          : ../data/zoo_outputs/zoo_meta_attack_zoo_001.json


## 6. Verify Attack + Inspect Monitor Signature

In [16]:
import pandas as pd

# This run's attack queries (gitignored, separate from baseline log)
with open(ATTACK_LOG_FILE) as f:
    attack_entries = [json.loads(l) for l in f]

# Normal traffic — last 500 frames of the committed baseline log
with open(LOG_FILE) as f:
    normal_entries = [e for e in (json.loads(l) for l in f) if e["source"] == "normal_video"][-500:]

def summarise(entries, label):
    deltas  = [e["delta_ms"] for e in entries if e["delta_ms"] is not None]
    phashes = [e["phash"] for e in entries]
    # Sample 50 consecutive pHash distances
    sample  = phashes[:50] if len(phashes) >= 2 else phashes
    dists   = [ih.hex_to_hash(sample[i]) - ih.hex_to_hash(sample[i+1])
               for i in range(min(len(sample)-1, 49))]
    print(f"\n{label}")
    print(f"  Queries          : {len(entries)}")
    print(f"  Mean delta_ms    : {np.mean(deltas):.1f}" if deltas else "  Mean delta_ms    : N/A")
    print(f"  pHash dist mean  : {np.mean(dists):.1f}" if dists else "  pHash dist       : N/A")
    print(f"  pHash dist range : {min(dists)}\u2013{max(dists)}" if dists else "")

summarise(normal_entries, "NORMAL (last 500 frames)")
summarise(attack_entries, f"ZOO ATTACK ({SESSION_ID})")

# Confidence progression
print("\nConf progression:")
for i, (c, q) in enumerate(zip(meta["conf_log"], meta["query_log"])):
    bar = "\u2588" * int(c * 40)
    print(f"  iter {i+1:3d} | q={q:5,} | {c:.3f} {bar}")



NORMAL (last 500 frames)
  Queries          : 500
  Mean delta_ms    : 6.8
  pHash dist mean  : 0.0
  pHash dist range : 0–0

ZOO ATTACK (attack_zoo_001)
  Queries          : 1400
  Mean delta_ms    : 55.8
  pHash dist mean  : 0.0
  pHash dist range : 0–0

Conf progression:
  iter   1 | q=    2 | 0.918 ████████████████████████████████████
  iter   2 | q=    4 | 0.918 ████████████████████████████████████
  iter   3 | q=    6 | 0.918 ████████████████████████████████████
  iter   4 | q=    8 | 0.918 ████████████████████████████████████
  iter   5 | q=   10 | 0.918 ████████████████████████████████████
  iter   6 | q=   12 | 0.917 ████████████████████████████████████
  iter   7 | q=   14 | 0.917 ████████████████████████████████████
  iter   8 | q=   16 | 0.917 ████████████████████████████████████
  iter   9 | q=   18 | 0.917 ████████████████████████████████████
  iter  10 | q=   20 | 0.917 ████████████████████████████████████
  iter  11 | q=   22 | 0.917 ███████████████████████████████████

In [ ]:
# Cell 15 — Verify detections: original vs adversarial
import cv2

adv_img_loaded = cv2.imread(str(adv_path))

orig_results = model(base_img, verbose=False)
adv_results  = model(adv_img_loaded, verbose=False)

def list_dets(results, label):
    print(f"\n{label} ({len(results[0].boxes)} detections):")
    for box in results[0].boxes:
        cls = int(box.cls[0])
        print(f"  {model.names[cls]:12s} conf={float(box.conf[0]):.3f}  bbox={[round(x,1) for x in box.xyxy[0].tolist()]}")

list_dets(orig_results, "ORIGINAL")
list_dets(adv_results, "ADVERSARIAL")

# Annotated side-by-side
orig_annotated = orig_results[0].plot()
adv_annotated  = adv_results[0].plot()
annotated_compare = np.hstack([orig_annotated, adv_annotated])
annotated_path = OUTPUT_DIR / f"zoo_annotated_compare_{SESSION_ID}.jpg"
cv2.imwrite(str(annotated_path), annotated_compare)
print(f"\nSaved: {annotated_path}")
